# Lecture 3.3 — Search Tools: Glob & Grep

**Course:** Build Production AI Agents with the Claude Agent SDK  
**Section 3:** Built-In Tools Deep Dive

---

In Lecture 3.1 we covered file tools (Read, Write, Edit). In Lecture 3.2 we covered shell tools (Bash, Monitor). Now we add two essential **search tools** that every agent working with a codebase needs:

| Tool | What it does |
|------|--------------|
| **Glob** | Finds **files** by name and extension pattern across a directory tree |
| **Grep** | Searches **inside files** for content matching a text or regex pattern |

We also include a brief regex primer — just enough to understand what Grep is doing and write effective search prompts.

This notebook is **fully self-contained** — fresh setup, fresh sample project, no carry-over from previous lectures.

In [1]:
# Install the SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 11.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

The `MODEL_NAME` variable below controls which Claude model powers the agents in this notebook.

You can change it to any model you have access to. For the latest available models, visit:  
https://platform.claude.com/docs/en/about-claude/models/overview

We use **claude-haiku-4-5** in this notebook — it is fast and cost-effective for these search demos.

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

---
## Creating the Sample Project

Before running any agent, we build a small but realistic Python project to search through:

```
search_demo/
  README.md
  notes.txt
  src/
    auth.py        2 TODOs, 3 functions
    utils.py       3 functions
    payments.py    2 TODOs, 2 functions
  tests/
    test_auth.py   2 test functions
```

We plant TODO comments and proper function definitions so our Glob and Grep demos have something meaningful to find.

In [4]:
import os

os.makedirs("/content/search_demo/src", exist_ok=True)
os.makedirs("/content/search_demo/tests", exist_ok=True)

with open("/content/search_demo/src/auth.py", "w") as f:
    f.write("# Authentication module\n")
    f.write("# TODO: Add password hashing\n")
    f.write("# TODO: Implement OAuth2 support\n\n")
    f.write("def login(user, password):\n")
    f.write("    \"\"\"Authenticate a user.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def logout(user):\n")
    f.write("    \"\"\"Log out the specified user.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def reset_password(user):\n")
    f.write("    \"\"\"Reset user password.\"\"\"\n")
    f.write("    pass\n")

with open("/content/search_demo/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def format_date(date):\n")
    f.write("    \"\"\"Format a date as YYYY-MM-DD.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def format_currency(amount):\n")
    f.write("    \"\"\"Format a number as currency.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def validate_email(email):\n")
    f.write("    \"\"\"Check if email address is valid.\"\"\"\n")
    f.write("    pass\n")

with open("/content/search_demo/src/payments.py", "w") as f:
    f.write("# Payments module\n")
    f.write("# TODO: Integrate Stripe payment gateway\n")
    f.write("# TODO: Add retry logic for failed transactions\n\n")
    f.write("def process_payment(amount, card):\n")
    f.write("    \"\"\"Process a payment transaction.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def refund_payment(transaction_id):\n")
    f.write("    \"\"\"Refund a payment transaction.\"\"\"\n")
    f.write("    pass\n")

with open("/content/search_demo/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write("    \"\"\"Test the login function.\"\"\"\n")
    f.write("    pass\n\n")
    f.write("def test_logout():\n")
    f.write("    \"\"\"Test the logout function.\"\"\"\n")
    f.write("    pass\n")

with open("/content/search_demo/README.md", "w") as f:
    f.write("# Search Demo Project\n")
    f.write("A demo project for the Glob and Grep lecture.\n")
    f.write("Contains authentication, payments, and utility modules.\n")

with open("/content/search_demo/notes.txt", "w") as f:
    f.write("Project Notes\n")
    f.write("=============\n")
    f.write("Action Items:\n")
    f.write("- Finish authentication module\n")
    f.write("- Write payment tests\n")
    f.write("- Update README\n")

print("Sample project created.")

Sample project created.


---
## Verifying What Was Created

Before any agent runs, let's print the complete directory structure and every file's content.

This is an important step: when Glob and Grep produce results, you can verify them against exactly what is in the files here.

In [5]:
import os

# Show directory structure
print("Project structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/search_demo"):
    level = root.replace("/content/search_demo", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

# Show all file contents
files_to_show = [
    "/content/search_demo/README.md",
    "/content/search_demo/notes.txt",
    "/content/search_demo/src/auth.py",
    "/content/search_demo/src/utils.py",
    "/content/search_demo/src/payments.py",
    "/content/search_demo/tests/test_auth.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Project structure:
search_demo/
  README.md
  notes.txt
  src/
    auth.py
    payments.py
    utils.py
  tests/
    test_auth.py

File contents:

--- /content/search_demo/README.md ---
# Search Demo Project
A demo project for the Glob and Grep lecture.
Contains authentication, payments, and utility modules.


--- /content/search_demo/notes.txt ---
Project Notes
Action Items:
- Finish authentication module
- Write payment tests
- Update README


--- /content/search_demo/src/auth.py ---
# Authentication module
# TODO: Add password hashing
# TODO: Implement OAuth2 support

def login(user, password):
    """Authenticate a user."""
    pass

def logout(user):
    """Log out the specified user."""
    pass

def reset_password(user):
    """Reset user password."""
    pass


--- /content/search_demo/src/utils.py ---
# Utility functions

def format_date(date):
    """Format a date as YYYY-MM-DD."""
    pass

def format_currency(amount):
    """Format a number as currency."""
    pass

def val

---
## Part 1 — A Brief Introduction to Regex

Grep searches inside files using **regex** (regular expressions) patterns. You don't need to master regex — these four patterns cover most real-world codebase searches:

| Pattern | What it matches | Example use |
|---------|-----------------|-------------|
| `TODO`  | The exact string "TODO" | Find all TODO comments |
| `^def`  | Lines **starting** with `def` | Find all function definitions |
| `.*`    | Any characters (wildcard) | Match anything |
| `\d+`   | One or more digits | Find numbers |

The `^` character anchors the pattern to the **start** of a line — so `^def` only matches lines that *begin* with `def`.

The agent handles the regex internally. You describe what you want to find in plain English. This is all the regex knowledge you need for this course.

In [6]:
import re

lines = [
    "def login(user, password):",
    "# TODO: Add password hashing",
    "    return True",
    "def logout(user):",
]

print("Lines that define functions (pattern: ^def):")
for line in lines:
    if re.match(r"^def", line):
        print(f"  MATCH: {line}")

print("\nLines with TODO comments (pattern: TODO):")
for line in lines:
    if re.search(r"TODO", line):
        print(f"  MATCH: {line}")

Lines that define functions (pattern: ^def):
  MATCH: def login(user, password):
  MATCH: def logout(user):

Lines with TODO comments (pattern: TODO):
  MATCH: # TODO: Add password hashing


---
## Part 2 — The Glob Tool

**Glob finds files by name and extension pattern.** It searches the directory tree and returns matching file paths.

Common glob patterns:

| Pattern | What it finds |
|---------|---------------|
| `**/*.py` | All Python files anywhere in the tree (`**` = any number of subdirectory levels) |
| `src/**/*.py` | All Python files inside `src/` |
| `tests/*.py` | Python files directly inside `tests/` |

You used Glob briefly in Section 2. This lecture goes deeper into what it can do.

**Key point:** Glob works at the level of *file names and paths* — it does not open or read file contents.

In [7]:
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="""
    Find all Python files in /content/search_demo.
    List them with their full paths, organised by folder.
    Also tell me how many Python files exist in total.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Glob"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Here are all the Python files in `/content/search_demo`, organized by folder:

## **src/** folder
- `/content/search_demo/src/auth.py`
- `/content/search_demo/src/payments.py`
- `/content/search_demo/src/utils.py`

## **tests/** folder
- `/content/search_demo/tests/test_auth.py`

**Total Python files: 4**


---
## Part 3 — The Grep Tool

**Grep searches inside files for content matching a pattern.** It reads file contents and returns the matching lines with their file locations.

When to use Grep:
- Finding all TODO comments across a project
- Locating where a specific function is defined
- Finding all usages of a variable or function call
- Searching for a string across many files at once

**Key point:** Grep operates at the level of *file contents* — it opens and reads files, unlike Glob which only looks at paths.

In [8]:
async for message in query(
    prompt="""
    Search all files in /content/search_demo for TODO comments.
    For each TODO found, show:
    1. The file it appears in
    2. The exact TODO text
    Present as a clean organised list.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Grep"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! Here's an organized list of all TODO comments found in `/content/search_demo`:

## TODO Comments Summary

### **src/payments.py**
1. Integrate Stripe payment gateway
2. Add retry logic for failed transactions

### **src/auth.py**
1. Add password hashing
2. Implement OAuth2 support

**Total: 4 TODOs across 2 files**


---
## Part 4 — Glob + Grep Together: A Powerful Combination

Glob and Grep are most powerful when used together. This mirrors a common developer workflow:

1. **Glob first** — narrow down: *which files do I care about?*
2. **Grep second** — find: *what is inside those files?*

With both tools available, **the agent decides the tool sequence itself** — you just describe the goal in plain English.

This is the highlight of the lecture: a single prompt that produces a complete codebase report.

In [9]:
async for message in query(
    prompt="""
    In the project at /content/search_demo:
    1. Use Glob to find all Python files
    2. Use Grep to search those Python files for all function definitions
       (lines starting with 'def')
    3. For each file, list the functions defined in it
    Present as a clean report showing each file and its functions.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Glob", "Grep"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! Here's a clean report of all Python files and their function definitions:

## Python Functions Report for `/content/search_demo`

### **src/auth.py**
- `login(user, password)`
- `logout(user)`
- `reset_password(user)`

### **src/payments.py**
- `process_payment(amount, card)`
- `refund_payment(transaction_id)`

### **src/utils.py**
- `format_date(date)`
- `format_currency(amount)`
- `validate_email(email)`

### **tests/test_auth.py**
- `test_login()`
- `test_logout()`

---

**Summary:** Found **4 Python files** containing a total of **10 functions** across the project.


---
## Part 5 — Glob vs Grep: When to Use Which

|  | **Glob** | **Grep** |
|--|----------|----------|
| **What it searches** | File names and paths | File contents |
| **Input** | A pattern like `**/*.py` | A regex like `^def` or `TODO` |
| **Output** | A list of matching file paths | Matching lines inside files |
| **Use when** | You need to *find files* | You need to *find content* |

**Use both together when:** you need specific content inside a specific subset of files — Glob narrows the files, Grep narrows the content.

---
## Summary

In this lecture we covered:

- **Glob** — finds files by name and extension pattern across an entire directory tree
- **Grep** — searches inside files for content matching a text or regex pattern
- **Regex primer** — `TODO`, `^def`, `.*`, `\d+` — all the regex you need for this course
- **Glob + Grep combined** — the agent autonomously ran Glob first, then Grep, and produced a complete function inventory from a single prompt

**Key insight:** Glob answers *which files*, Grep answers *what is inside them*. Together they give your agent the ability to navigate any codebase just as a senior developer would — but faster and at any scale.

---

### Coming Up: Lecture 3.4 — Web Tools: WebSearch & WebFetch

We take the agent off your local filesystem entirely and connect it to the internet. Your agent will be able to search the web for current information and fetch and parse full web pages — all from a natural-language prompt.